In [16]:
## Import libraries needed
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


## Dataset link
REPO = "hf://datasets/DenyTranDFW/HYUNDAI_ABS_FUNDING_LLC_1260125"

## Relevant files in the dataset
## These files cover a period of 7 months of reporting data for loans
files = [
    "000114420417065485/hca-autoloanex102ald.parquet",  # November 2017
    "000114420418004231/hca-autoloanex102ald.parquet",  # December 2017
    "000114420418011202/hca-autoloanex102ald.parquet",  # January 2018
    "000114420418017122/hca-autoloanex102ald.parquet",  # February 2018
    "000114420418023063/hca-autoloanex102ald.parquet",  # March 2018
    "000114420418031297/hca-autoloanex102ald.parquet",  # April 2018
    "000114420418035771/hca-autoloanex102ald.parquet",  # May 2018
]

## Select relevant columns
cols = [
    "assetNumber", ## Unique Loan ID
    "originationDate",
    "reportingPeriodEndingDate", ## month a row is from
    "obligorCreditScore",
    "paymentToIncomePercentage",
    "originalInterestRatePercentage",
    "coObligorIndicator", ## do they have a cosigner?
    "originalLoanAmount",
    "originalLoanTerm",
    "vehicleNewUsedCode",
    "currentDelinquencyStatus"
]

In [17]:
## Combine all of the files into one dataset
monthly = []

for f in files:
    df = pd.read_parquet(REPO + "/" + f, columns = cols)
    monthly.append(df)

loans = pd.concat(monthly, ignore_index=True)
print(loans.shape)
loans.head()

(498918, 11)


,assetNumber,originationDate,reportingPeriodEndingDate,obligorCreditScore,paymentToIncomePercentage,originalInterestRatePercentage,coObligorIndicator,originalLoanAmount,originalLoanTerm,vehicleNewUsedCode,currentDelinquencyStatus
0,17A011677,2016-08-01,2017-11-30,798,0.052105,0.0090,False,21136.48,61,1,0.0
1,17A019616,2016-10-01,2017-11-30,624,0.066555,0.1189,False,25606.95,73,1,0.0
2,17A011730,2016-08-01,2017-11-30,763,0.040529,0.0090,False,23761.05,62,1,0.0
3,17A069214,2016-10-01,2017-11-30,722,0.034839,0.0299,False,11837.08,73,1,0.0
4,17A014113,2016-09-01,2017-11-30,755,0.050242,0.0329,False,19363.52,76,1,0.0


In [18]:
## Changes the data type of reportingPeriodEndingDate from text to actual date type
loans["reportingPeriodEndingDate"] = pd.to_datetime(loans["reportingPeriodEndingDate"])

# Converts currentDeliquencyStatus to a number
## "coerce" turns any non-number into a nan
loans["currentDelinquencyStatus"] = pd.to_numeric(loans["currentDelinquencyStatus"],errors="coerce")


In [19]:
## This cells only keeps the rows that show up in all 7 months

## Count how many months each loan appears
months_per_loan = loans["assetNumber"].value_counts()

## Grabs the loans that have all 7 months
full_loans = months_per_loan[months_per_loan == 7].index

## Keeps only the loans in full_loans
loans = loans[loans["assetNumber"].isin(full_loans)]

print("Loans left: ", loans["assetNumber"].nunique())

Loans left:  71274


In [20]:
## Removes any loans with na values in currentDelinquency

bad_loans = loans.loc[loans["currentDelinquencyStatus"].isna(), "assetNumber"].unique()

loans = loans[~loans["assetNumber"].isin(bad_loans)]

print("Loans left:", loans["assetNumber"].nunique())

Loans left: 70745


In [21]:
## Groups the rows by assetNumber and grabs the highest delinquency for that loan
worst_status = loans.groupby("assetNumber")["currentDelinquencyStatus"].max()

## Creates a series with 0/1 values for delinquency
## Will get added to the final dataset later
ever_delinquent = (worst_status >= 30).astype(int) ## True = 1, False = 0

print("0 = Never delinquent\n1 = 30+ days delinquent at least once")
ever_delinquent.value_counts()


0 = Never delinquent
1 = 30+ days delinquent at least once


currentDelinquencyStatus
0    67986
1     2759
Name: count, dtype: int64

In [22]:
## Sorts the loans by assetnumber (first -> last), then by month
loans = loans.sort_values(["assetNumber", "reportingPeriodEndingDate"])

# Drops any duplicates so that each loan only appears once
data = loans.drop_duplicates("assetNumber", keep="first").copy()

## Drops these columns as features because they change every month
data = data.drop(columns=["reportingPeriodEndingDate", "currentDelinquencyStatus"])

## Add column ever delinquent. map() matches the assetNumber with ever_delinquent
data["everDelinquent"] = data["assetNumber"].map(ever_delinquent)

print(data.shape)
data.head()

(70745, 10)


,assetNumber,originationDate,obligorCreditScore,paymentToIncomePercentage,originalInterestRatePercentage,coObligorIndicator,originalLoanAmount,originalLoanTerm,vehicleNewUsedCode,everDelinquent
44266,17A000001,2012-01-01,656,0.080969,0.0829,False,32877.08,73,1,0
16236,17A000002,2012-01-01,672,0.172831,0.0989,False,27000.00,73,1,0
12599,17A000003,2012-01-01,675,0.063686,0.0490,True,22166.45,73,1,0
52350,17A000004,2012-01-01,623,0.094049,0.1029,False,21300.97,73,1,1
64935,17A000005,2012-01-01,687,0.103191,0.0390,False,32643.97,73,1,0


In [23]:
## Noticed some of the rows had 0 as a credit score, but we'd rather treat that as
## a nan and handle it with the imputer later by changing nan values to median
## since credit scores don't go that low
data["obligorCreditScore"] = data["obligorCreditScore"].replace(0, np.nan)
print(data["obligorCreditScore"].isna().sum())


787


In [24]:
## Check the ratio of delinquent and never delinquent
print(data["everDelinquent"].value_counts(normalize=True))


data.to_csv("loan_model_dataset.csv", index=False)

everDelinquent
0    0.961001
1    0.038999
Name: proportion, dtype: float64
